# DSSM baseline

two-tower модель: user tower — average pooling эмбеддингов истории + MLP, item tower — MLP над эмбеддингом товара, in-batch negatives. использует обычные ID товаров, без семантических кодов

## imports & install

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

## data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/pet_proj_recomendashki'
seq_df = pd.read_parquet(f'{base_path}/data/sequences_final')
with open(f'{base_path}/data/items_to_idx.json') as f:
    num_items = len(json.load(f))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

eval_df = seq_df.sample(n=1000, random_state=42)
val_df = seq_df.sample(n=1000, random_state=0)
print(num_items, device)

Mounted at /content/drive
26354 cuda


обучающие примеры — все префиксы `train_hist`: по истории `hist[:i]` предсказываем `hist[i]`

In [ ]:
class DSSMDatasetRich(Dataset):
    def __init__(self, seq_df, min_history=1):
        self.samples = []
        for _, row in seq_df.iterrows():
            hist = [int(i) for i in row['train_hist']]
            for i in range(min_history, len(hist)):
                self.samples.append((hist[:i], hist[i]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def dssm_collate_fn(batch):
    histories = [torch.tensor(h, dtype=torch.long) for h, t in batch]
    targets = torch.tensor([t for h, t in batch], dtype=torch.long)
    padded_histories = pad_sequence(histories, batch_first=True, padding_value=-1)
    return padded_histories, targets


dataset = DSSMDatasetRich(seq_df)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True, collate_fn=dssm_collate_fn)
print(len(dataset))

533148


## архитектура

In [ ]:
class DSSM(nn.Module):
    def __init__(self, num_items, embed_dim=64, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items, embed_dim)
        self.user_mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim)
        )

    def encode_user(self, history_ids, pad_value=-1):
        mask = (history_ids != pad_value).float().unsqueeze(-1)
        safe_ids = history_ids.clamp(min=0)
        embeds = self.item_embedding(safe_ids) * mask
        pooled = embeds.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.user_mlp(pooled)

    def encode_item(self, item_ids):
        return self.item_mlp(self.item_embedding(item_ids))

    def forward(self, history_ids, item_ids):
        return self.encode_user(history_ids), self.encode_item(item_ids)

## metrics

ранг правильного товара среди всех 26 354, Recall@k и NDCG@k считаются из рангов

In [ ]:
def dssm_ranks(model, seq_df, num_items, device, hist_col, target_col):
    model.eval()
    ranks = []
    with torch.no_grad():
        all_item_vecs = model.encode_item(torch.arange(num_items, device=device))
        for _, row in seq_df.iterrows():
            history_tensor = torch.tensor([[int(i) for i in row[hist_col]]], dtype=torch.long, device=device)
            user_vec = model.encode_user(history_tensor)
            scores = (user_vec @ all_item_vecs.T).squeeze(0)
            true_item_id = int(row[target_col])
            ranks.append((scores > scores[true_item_id]).sum().item() + 1)
    return np.array(ranks)


def ranks_to_metrics(ranks, k_list=[5, 10, 50, 100]):
    results = {}
    for k in k_list:
        results[f'Recall@{k}'] = np.mean(ranks <= k)
        results[f'NDCG@{k}'] = np.mean(np.where(ranks <= k, 1 / np.log2(ranks + 1), 0.0))
    return results

## обучение

early stopping по медианному рангу на валидации (`train_hist` → `val`), проверка каждые 5 эпох

In [ ]:
def train_dssm_with_early_stopping(model, dataloader, optimizer, device, val_df, num_items,
                                   max_epochs=200, eval_every=5, patience=3):
    hist = {'loss': [], 'val_epoch': [], 'val_median_rank': []}
    best_rank = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0
        n_batch = 0
        for history_ids, target_ids in dataloader:
            n_batch += 1
            history_ids = history_ids.to(device)
            target_ids = target_ids.to(device)
            optimizer.zero_grad()
            user_vec, item_vec = model(history_ids, target_ids)
            scores = user_vec @ item_vec.T
            labels = torch.arange(scores.shape[0], device=device)
            loss = F.cross_entropy(scores, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / n_batch
        hist['loss'].append(avg_loss)

        if (epoch + 1) % eval_every == 0:
            median_rank = float(np.median(dssm_ranks(model, val_df, num_items, device, 'train_hist', 'val')))
            hist['val_epoch'].append(epoch + 1)
            hist['val_median_rank'].append(median_rank)
            median_rank = int(median_rank)
            print(f"epoch {epoch+1} | loss: {avg_loss:.4f} | median VAL rank: {median_rank}")

            if median_rank < best_rank:
                best_rank = median_rank
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f"early stopping на эпохе {epoch+1}, лучший val ранг: {best_rank}")
                    break

    model.load_state_dict(best_state)
    return model, hist

In [ ]:
model_dssm = DSSM(num_items=num_items, embed_dim=64, hidden_dim=128, dropout=0.2).to(device)
optimizer_dssm = torch.optim.Adam(model_dssm.parameters(), lr=1e-3, weight_decay=1e-5)

model_dssm, hist_dssm = train_dssm_with_early_stopping(
    model_dssm, dataloader, optimizer_dssm, device, val_df, num_items
)

epoch 5 | loss: 4.5553 | median VAL rank: 2166
epoch 10 | loss: 4.2098 | median VAL rank: 1538
epoch 15 | loss: 4.0482 | median VAL rank: 1416
epoch 20 | loss: 3.9250 | median VAL rank: 1422
epoch 25 | loss: 3.8205 | median VAL rank: 1340
epoch 30 | loss: 3.7355 | median VAL rank: 1247
epoch 35 | loss: 3.6609 | median VAL rank: 1254
epoch 40 | loss: 3.5931 | median VAL rank: 1266
epoch 45 | loss: 3.5343 | median VAL rank: 1296
early stopping на эпохе 45, лучший val ранг: 1247


## тест

In [ ]:
test_ranks = dssm_ranks(model_dssm, eval_df, num_items, device, 'test_hist', 'test')
print(ranks_to_metrics(test_ranks))
print("медианный ранг:", np.median(test_ranks), "| % в топ-1000:", np.mean(test_ranks <= 1000) * 100)

{'Recall@5': np.float64(0.015), 'NDCG@5': np.float64(0.00941524229750988), 'Recall@10': np.float64(0.022), 'NDCG@10': np.float64(0.011614872219822452), 'Recall@50': np.float64(0.079), 'NDCG@50': np.float64(0.023669532995229253), 'Recall@100': np.float64(0.131), 'NDCG@100': np.float64(0.03203827656600014)}
медианный ранг: 1674.0 | % в топ-1000: 39.7


## save

In [ ]:
torch.save({
    'model_state_dict': model_dssm.state_dict(),
    'hist': hist_dssm,
    'config': {'num_items': num_items, 'embed_dim': 64, 'hidden_dim': 128, 'dropout': 0.2},
}, f'{base_path}/checkpoints/dssm_final.pt')